# Notebook 17 — Sepsis Progression Target Definition (Fixed)

Rebuilt clean, no dead debug cells. Adds current SOFA score, SOFA trajectory, SOFA subscores, and lab forward-fill/time-since-lab features that were missing from the original pipeline.

## 1. Imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
pd.set_option("display.max_columns", None)
print("Libraries imported")

Libraries imported


## 2. Load raw hourly timeseries + cohort

In [2]:
timeseries = pd.read_csv(
    "../data/hf_dataset/sepsis_timeseries_hourly.csv"
)
cohort = pd.read_csv(
    "../data/hf_dataset/sepsis_cohort.csv"
)
print("Timeseries shape:", timeseries.shape)
print("Cohort shape:", cohort.shape)

Timeseries shape: (851717, 64)
Cohort shape: (32971, 12)


In [3]:
timeseries["starttime"] = pd.to_datetime(timeseries["starttime"])
timeseries["endtime"] = pd.to_datetime(timeseries["endtime"])
cohort["sepsis_time"] = pd.to_datetime(cohort["sepsis_time"])
cohort["icu_intime"] = pd.to_datetime(cohort["icu_intime"])
cohort["icu_outtime"] = pd.to_datetime(cohort["icu_outtime"])

print("Datetime conversion complete")

Datetime conversion complete


## 3. Build `model` table: timeseries + sepsis_time + hours_from_sepsis

In [5]:
model = timeseries.merge(
    cohort[["stay_id", "sepsis_time"]],
    on="stay_id",
    how="left"
)

model["hours_from_sepsis"] = (
    model["starttime"] - model["sepsis_time"]
).dt.total_seconds() / 3600

print("Model shape:", model.shape)
print(
    model[
        ["stay_id", "starttime", "sepsis_time", "hours_from_sepsis", "sofa_score"]
    ].head()
)

Model shape: (851717, 66)
    stay_id           starttime         sepsis_time  hours_from_sepsis  \
0  30000484 2136-01-14 17:23:32 2136-01-14 19:00:00          -1.607778   
1  30000484 2136-01-14 18:23:32 2136-01-14 19:00:00          -0.607778   
2  30000484 2136-01-14 19:23:32 2136-01-14 19:00:00           0.392222   
3  30000484 2136-01-14 20:23:32 2136-01-14 19:00:00           1.392222   
4  30000484 2136-01-14 21:23:32 2136-01-14 19:00:00           2.392222   

   sofa_score  
0         NaN  
1         3.0  
2         1.0  
3         1.0  
4         0.0  


## 4. Post-sepsis observations only

In [6]:
post_sepsis = model[
    model["hours_from_sepsis"] >= 0
].copy()

print("Post-sepsis observations:", len(post_sepsis))
print("Post-sepsis ICU stays:", post_sepsis["stay_id"].nunique())

Post-sepsis observations: 605121
Post-sepsis ICU stays: 31333


In [7]:
duration = (
    post_sepsis
    .groupby("stay_id")["hours_from_sepsis"]
    .max()
)
print(duration.describe(
    percentiles=[0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
))

count    31333.000000
mean        18.809560
std          4.562092
min          0.000000
10%         12.192444
25%         17.383333
50%         20.501944
75%         22.000000
90%         22.483333
95%         22.686944
99%         22.954178
max         40.856111
Name: hours_from_sepsis, dtype: float64


## 5. Build 6-hour-ahead SOFA target

In [8]:
sofa_data = post_sepsis[
    ["stay_id", "hours_from_sepsis", "sofa_score"]
].dropna(subset=["sofa_score"]).copy()

sofa_data = sofa_data.sort_values(["stay_id", "hours_from_sepsis"])

print("SOFA observations:", len(sofa_data))
print("ICU stays:", sofa_data["stay_id"].nunique())
print("\nSOFA range:")
print(sofa_data["sofa_score"].describe())

SOFA observations: 583370
ICU stays: 31285

SOFA range:
count    583370.000000
mean          1.468281
std           1.852888
min           0.000000
25%           0.000000
50%           1.000000
75%           3.000000
max          18.000000
Name: sofa_score, dtype: float64


In [9]:
current = sofa_data[
    ["stay_id", "hours_from_sepsis", "sofa_score"]
].copy()

future = sofa_data[
    ["stay_id", "hours_from_sepsis", "sofa_score"]
].copy()

future = future.rename(
    columns={
        "hours_from_sepsis": "future_hours",
        "sofa_score": "future_sofa_6h"
    }
)

current["target_future_hour"] = current["hours_from_sepsis"] + 6

In [10]:
sofa_progression = pd.merge_asof(
    current.sort_values("target_future_hour"),
    future.sort_values("future_hours"),
    left_on="target_future_hour",
    right_on="future_hours",
    by="stay_id",
    direction="nearest",
    tolerance=1.0
)
print(sofa_progression.head())

    stay_id  hours_from_sepsis  sofa_score  target_future_hour  future_hours  \
0  35438780                0.0         4.0                 6.0           6.0   
1  33631920                0.0         0.0                 6.0           6.0   
2  39709334                0.0         1.0                 6.0           6.0   
3  32279258                0.0         1.0                 6.0           6.0   
4  38634786                0.0         1.0                 6.0           6.0   

   future_sofa_6h  
0             1.0  
1             0.0  
2             1.0  
3             0.0  
4             1.0  


In [11]:
sofa_progression["delta_sofa_6h"] = (
    sofa_progression["future_sofa_6h"]
    - sofa_progression["sofa_score"]
)

print(sofa_progression["delta_sofa_6h"].value_counts().sort_index())

delta_sofa_6h
-17.0         1
-16.0         1
-15.0         1
-14.0         3
-13.0        17
-12.0        23
-11.0        80
-10.0       108
-9.0        218
-8.0        535
-7.0        927
-6.0       1755
-5.0       3127
-4.0       8001
-3.0      18821
-2.0      23480
-1.0      64121
 0.0     208798
 1.0      55558
 2.0      17579
 3.0      13463
 4.0       6378
 5.0       2430
 6.0       1599
 7.0        918
 8.0        505
 9.0        220
 10.0       151
 11.0        80
 12.0        30
 13.0        14
 14.0         6
Name: count, dtype: int64


In [12]:
def classify_progression(delta):
    if delta < 0:
        return 0       # Improving
    elif delta == 0:
        return 1       # Stable
    else:
        return 2       # Deteriorating


sofa_progression["progression_class"] = (
    sofa_progression["delta_sofa_6h"]
    .apply(classify_progression)
)

print(sofa_progression["progression_class"].value_counts().sort_index())

progression_class
0    121219
1    208798
2    253353
Name: count, dtype: int64


In [13]:
sofa_target = sofa_progression[
    sofa_progression["delta_sofa_6h"].notna()
].copy()

print("Total valid target rows:", len(sofa_target))
print("Missing targets:", sofa_target["delta_sofa_6h"].isna().sum())
print("\nClass distribution:")
print(sofa_target["progression_class"].value_counts().sort_index())
print("\nNumber of unique ICU stays:", sofa_target["stay_id"].nunique())

Total valid target rows: 428948
Missing targets: 0

Class distribution:
progression_class
0    121219
1    208798
2     98931
Name: count, dtype: int64

Number of unique ICU stays: 30434


## 6. Load enhanced feature dataset (from Notebook 11)

In [14]:
feature_dataset = pd.read_csv(
    "../data/processed/enhanced_feature_dataset.csv"
)

print("Feature dataset shape:", feature_dataset.shape)
print("\nFeature columns:")
print(feature_dataset.columns.tolist())

Feature dataset shape: (851717, 129)

Feature columns:
['stay_id', 'hour', 'heart_rate', 'resp_rate', 'temperature', 'sbp', 'dbp', 'mbp', 'spo2', 'gcs', 'creatinine', 'bun', 'urineoutput_last', 'urineoutput_sum', 'urineoutput_24hr', 'wbc', 'hemoglobin', 'hematocrit', 'platelet', 'bands', 'sodium', 'potassium', 'chloride', 'bicarbonate', 'calcium', 'magnesium', 'aniongap', 'albumin', 'bilirubin_total', 'bilirubin_max', 'inr', 'pt', 'ptt', 'crp', 'lactate', 'pao2fio2ratio_novent', 'pao2fio2ratio_vent', 'glucose_lab', 'shock_index', 'map_calculated', 'bun_creatinine_ratio', 'spo2_deficit', 'heart_rate_prev', 'heart_rate_delta', 'resp_rate_prev', 'resp_rate_delta', 'temperature_prev', 'temperature_delta', 'sbp_prev', 'sbp_delta', 'mbp_prev', 'mbp_delta', 'spo2_prev', 'spo2_delta', 'gcs_prev', 'gcs_delta', 'heart_rate_roll3_mean', 'heart_rate_roll6_mean', 'sofa_score_x', 'sofa_score_y', 'sofa_score', 'sofa_prev', 'sofa_delta_1h', 'sofa_roll3_mean', 'sofa_roll3_max', 'sofa_roll3_min', 'sofa_

## 7. Map sepsis-relative time to ICU hour (for merging target onto features)

In [15]:
time_mapping = (
    model[
        ["stay_id", "hour", "starttime", "sepsis_time", "hours_from_sepsis"]
    ]
    .drop_duplicates(subset=["stay_id", "hour"])
    .copy()
)

print("Time mapping shape:", time_mapping.shape)

Time mapping shape: (851717, 5)


In [16]:
target_sorted = sofa_target.sort_values(
    ["hours_from_sepsis", "stay_id"]
).reset_index(drop=True)

mapping_sorted = time_mapping.sort_values(
    ["hours_from_sepsis", "stay_id"]
).reset_index(drop=True)

target_with_hour = pd.merge_asof(
    target_sorted,
    mapping_sorted[["stay_id", "hours_from_sepsis", "hour"]],
    on="hours_from_sepsis",
    by="stay_id",
    direction="nearest",
    tolerance=0.51
)

print("Target with feature hour shape:", target_with_hour.shape)
print(
    "Target rows without matched feature hour:",
    target_with_hour["hour"].isna().sum()
)
print(
    "Percentage matched:",
    target_with_hour["hour"].notna().mean() * 100
)

Target with feature hour shape: (428948, 9)
Target rows without matched feature hour: 0
Percentage matched: 100.0


## 8. FIX — add current SOFA total score onto feature_dataset

This was missing entirely before. Target is `future_sofa - current_sofa`, so the model needs to know current_sofa.

In [21]:
sofa_current = model[["stay_id", "hour", "sofa_score"]].copy()
sofa_current = sofa_current.drop_duplicates(subset=["stay_id", "hour"])

feature_dataset = feature_dataset.merge(
    sofa_current,
    on=["stay_id", "hour"],
    how="left"
)

print("Feature dataset shape after SOFA merge:", feature_dataset.shape)
print("Missing sofa_score after merge:", feature_dataset["sofa_score"].isna().sum())

MergeError: Passing 'suffixes' which cause duplicate columns {'sofa_score_x', 'sofa_score_y'} is not allowed.

## 9. FIX — SOFA trajectory features

In [19]:
feature_dataset = feature_dataset.sort_values(["stay_id", "hour"])

grp = feature_dataset.groupby("stay_id")["sofa_score"]

feature_dataset["sofa_prev"] = grp.shift(1)
feature_dataset["sofa_delta_1h"] = feature_dataset["sofa_score"] - feature_dataset["sofa_prev"]

feature_dataset["sofa_roll3_mean"] = grp.transform(lambda x: x.rolling(3, min_periods=1).mean())
feature_dataset["sofa_roll3_max"]  = grp.transform(lambda x: x.rolling(3, min_periods=1).max())
feature_dataset["sofa_roll3_min"]  = grp.transform(lambda x: x.rolling(3, min_periods=1).min())
feature_dataset["sofa_roll6_mean"] = grp.transform(lambda x: x.rolling(6, min_periods=1).mean())
feature_dataset["sofa_roll6_max"]  = grp.transform(lambda x: x.rolling(6, min_periods=1).max())

print(
    feature_dataset[
        ["stay_id", "hour", "sofa_score", "sofa_prev", "sofa_delta_1h",
         "sofa_roll3_mean", "sofa_roll6_mean"]
    ].head(15)
)

     stay_id  hour  sofa_score  sofa_prev  sofa_delta_1h  sofa_roll3_mean  \
0   30000484     0         NaN        NaN            NaN              NaN   
1   30000484     1         3.0        NaN            NaN         3.000000   
2   30000484     2         1.0        3.0           -2.0         2.000000   
3   30000484     3         1.0        1.0            0.0         1.666667   
4   30000484     4         0.0        1.0           -1.0         0.666667   
5   30000484     5         1.0        0.0            1.0         0.666667   
6   30000484     6         1.0        1.0            0.0         0.666667   
7   30000484     7         2.0        1.0            1.0         1.333333   
8   30000484     8         1.0        2.0           -1.0         1.333333   
9   30000484     9         1.0        1.0            0.0         1.333333   
10  30000484    10         1.0        1.0            0.0         1.000000   
11  30000484    11         4.0        1.0            3.0         2.000000   

## 10. FIX — SOFA subscores (respiration/coagulation/liver/cardiovascular/cns/renal)

`sepsis_sofa_hourly_clock.csv` uses different hour binning than `feature_dataset`, so match on actual timestamp, not on the `hour` integer.

In [20]:
sofa_hourly = pd.read_csv("../data/hf_dataset/sepsis_sofa_hourly_clock.csv")
sofa_hourly["starttime"] = pd.to_datetime(sofa_hourly["starttime"])

fd_time = model[["stay_id", "hour", "starttime"]].drop_duplicates(subset=["stay_id", "hour"])
feature_dataset = feature_dataset.merge(fd_time, on=["stay_id", "hour"], how="left")

sub_cols = ["respiration", "coagulation", "liver", "cardiovascular", "cns", "renal"]

left = feature_dataset.sort_values("starttime")
right = sofa_hourly[["stay_id", "starttime"] + sub_cols].sort_values("starttime")

feature_dataset = pd.merge_asof(
    left, right,
    on="starttime",
    by="stay_id",
    direction="nearest",
    tolerance=pd.Timedelta("1h")
)

print("Missing subscore rate:")
print(feature_dataset[sub_cols].isna().mean())

feature_dataset = feature_dataset.drop(columns=["starttime"])

Missing subscore rate:


KeyError: "None of [Index(['respiration', 'coagulation', 'liver', 'cardiovascular', 'cns',\n       'renal'],\n      dtype='str')] are in the [columns]"

## 11. FIX — forward-fill labs + hours-since-last-lab

Raw lab columns are 90-99% missing (only present the exact hour drawn). Forward-fill the last known value per stay and add a recency feature.

In [19]:
lab_cols = [
    "creatinine", "bun", "wbc", "hemoglobin", "hematocrit", "platelet", "bands",
    "sodium", "potassium", "chloride", "bicarbonate", "calcium", "magnesium",
    "aniongap", "albumin", "bilirubin_total", "bilirubin_max", "inr", "pt", "ptt",
    "crp", "lactate", "glucose_lab", "pao2fio2ratio_novent", "pao2fio2ratio_vent"
]

feature_dataset = feature_dataset.sort_values(["stay_id", "hour"])

for col in lab_cols:
    g = feature_dataset.groupby("stay_id")[col]
    feature_dataset[f"{col}_ffill"] = g.ffill()

    mask = feature_dataset[col].notna()
    hr_if_present = feature_dataset["hour"].where(mask)
    hr_if_present = feature_dataset.groupby("stay_id")["hour"].transform(
        lambda h: h.where(mask.loc[h.index]).ffill()
    )
    feature_dataset[f"{col}_hours_since"] = feature_dataset["hour"] - hr_if_present

print("New forward-filled missing rate (top 10, should be far lower than raw):")
print(
    feature_dataset[[f"{c}_ffill" for c in lab_cols]]
    .isna().mean().sort_values(ascending=False).head(10)
)

C:\Users\prajin\AppData\Local\Temp\ipykernel_19476\4089190352.py:12: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  feature_dataset[f"{col}_ffill"] = g.ffill()
C:\Users\prajin\AppData\Local\Temp\ipykernel_19476\4089190352.py:19: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  feature_dataset[f"{col}_hours_since"] = feature_dataset["hour"] - hr_if_present
C:\Users\prajin\AppData\Local\Temp\ipykernel_19476\4089190352.py:12: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many 

New forward-filled missing rate (top 10, should be far lower than raw):
crp_ffill                     0.982951
bands_ffill                   0.890259
pao2fio2ratio_novent_ffill    0.882745
albumin_ffill                 0.731682
pao2fio2ratio_vent_ffill      0.644189
bilirubin_total_ffill         0.587864
bilirubin_max_ffill           0.587725
lactate_ffill                 0.441737
ptt_ffill                     0.264200
inr_ffill                     0.260439
dtype: float64


C:\Users\prajin\AppData\Local\Temp\ipykernel_19476\4089190352.py:19: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  feature_dataset[f"{col}_hours_since"] = feature_dataset["hour"] - hr_if_present


## 12. Save enhanced feature dataset with fixes

In [20]:
feature_dataset.to_csv("../data/processed/enhanced_feature_dataset.csv", index=False)
print("Saved enhanced_feature_dataset.csv with SOFA + lab fixes:", feature_dataset.shape)

Saved enhanced_feature_dataset.csv with SOFA + lab fixes: (851717, 122)


## 13. Merge fixed features with progression target

In [21]:
target_for_model = target_with_hour[
    [
        "stay_id",
        "hour",
        "progression_class",
        "delta_sofa_6h",
        "future_sofa_6h"
    ]
].copy()

training_dataset = feature_dataset.merge(
    target_for_model,
    on=["stay_id", "hour"],
    how="inner"
)

print("Training dataset shape:", training_dataset.shape)
print("\nProgression class distribution:")
print(training_dataset["progression_class"].value_counts().sort_index())

Training dataset shape: (428948, 125)

Progression class distribution:
progression_class
0    121219
1    208798
2     98931
Name: count, dtype: int64


## 14. Leakage check

In [22]:
leakage_columns = [
    "progression_class",
    "delta_sofa_6h",
    "future_sofa_6h"
]

feature_columns = [
    col for col in training_dataset.columns
    if col not in leakage_columns
]

print("Number of candidate features:", len(feature_columns))
print("\nChecking whether target columns are present:")
for col in leakage_columns:
    print(col, "->", col in training_dataset.columns)

Number of candidate features: 122

Checking whether target columns are present:
progression_class -> True
delta_sofa_6h -> True
future_sofa_6h -> True


## 15. Missing-value check + duplicate check

In [23]:
missing = training_dataset[feature_columns].isna().sum().sort_values(ascending=False)
missing_percent = missing / len(training_dataset) * 100
missing_table = pd.DataFrame({"missing_count": missing, "missing_percent": missing_percent})
print(missing_table.head(20))

                      missing_count  missing_percent
crp                          428592        99.917006
bands                        426305        99.383841
pao2fio2ratio_novent         424640        98.995682
crp_ffill                    422912        98.592836
crp_hours_since              422912        98.592836
albumin                      422334        98.458088
bilirubin_max                415762        96.925968
bilirubin_total              415749        96.922937
liver                        415708        96.913379
gcs_delta                    405286        94.483714
inr                          403204        93.998340
pt                           403199        93.997174
ptt                          402598        93.857064
pao2fio2ratio_vent           396880        92.524036
lactate                      396499        92.435214
calcium                      394226        91.905313
respiration                  392547        91.513890
wbc                          392350        91.

In [24]:
duplicates = training_dataset.duplicated(subset=["stay_id", "hour"]).sum()
print("Duplicate stay_id + hour rows:", duplicates)

Duplicate stay_id + hour rows: 0


## 16. Final feature list + target

In [25]:
excluded_columns = [
    "stay_id",
    "progression_class",
    "delta_sofa_6h",
    "future_sofa_6h"
]

feature_columns = [
    col for col in training_dataset.columns
    if col not in excluded_columns
]

target_column = "progression_class"

print("Number of model features:", len(feature_columns))
print("\nModel features:")
print(feature_columns)

Number of model features: 121

Model features:
['hour', 'heart_rate', 'resp_rate', 'temperature', 'sbp', 'dbp', 'mbp', 'spo2', 'gcs', 'creatinine', 'bun', 'urineoutput_last', 'urineoutput_sum', 'urineoutput_24hr', 'wbc', 'hemoglobin', 'hematocrit', 'platelet', 'bands', 'sodium', 'potassium', 'chloride', 'bicarbonate', 'calcium', 'magnesium', 'aniongap', 'albumin', 'bilirubin_total', 'bilirubin_max', 'inr', 'pt', 'ptt', 'crp', 'lactate', 'pao2fio2ratio_novent', 'pao2fio2ratio_vent', 'glucose_lab', 'shock_index', 'map_calculated', 'bun_creatinine_ratio', 'spo2_deficit', 'heart_rate_prev', 'heart_rate_delta', 'resp_rate_prev', 'resp_rate_delta', 'temperature_prev', 'temperature_delta', 'sbp_prev', 'sbp_delta', 'mbp_prev', 'mbp_delta', 'spo2_prev', 'spo2_delta', 'gcs_prev', 'gcs_delta', 'heart_rate_roll3_mean', 'heart_rate_roll6_mean', 'sofa_score', 'sofa_prev', 'sofa_delta_1h', 'sofa_roll3_mean', 'sofa_roll3_max', 'sofa_roll3_min', 'sofa_roll6_mean', 'sofa_roll6_max', 'respiration', 'coag

In [26]:
non_numeric = training_dataset[feature_columns].select_dtypes(exclude=np.number).columns.tolist()
print("Non-numeric features:", non_numeric)

Non-numeric features: []


## 17. Patient-level train / validation / test split

In [27]:
from sklearn.model_selection import GroupShuffleSplit

X = training_dataset[feature_columns].copy()
y = training_dataset[target_column].copy()
groups = training_dataset["stay_id"]

print("Total observations:", len(training_dataset))
print("Unique ICU stays:", groups.nunique())

gss1 = GroupShuffleSplit(n_splits=1, test_size=0.30, random_state=42)
train_idx, temp_idx = next(gss1.split(X, y, groups=groups))

X_train = X.iloc[train_idx].copy()
y_train = y.iloc[train_idx].copy()
X_temp = X.iloc[temp_idx].copy()
y_temp = y.iloc[temp_idx].copy()
groups_temp = groups.iloc[temp_idx]

gss2 = GroupShuffleSplit(n_splits=1, test_size=0.50, random_state=42)
val_idx, test_idx = next(gss2.split(X_temp, y_temp, groups=groups_temp))

X_val = X_temp.iloc[val_idx].copy()
y_val = y_temp.iloc[val_idx].copy()
X_test = X_temp.iloc[test_idx].copy()
y_test = y_temp.iloc[test_idx].copy()

print("\nDataset sizes:")
print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

print("\nUnique stays:")
print("Train:", groups.iloc[train_idx].nunique())
print("Validation:", groups_temp.iloc[val_idx].nunique())
print("Test:", groups_temp.iloc[test_idx].nunique())

Total observations: 428948
Unique ICU stays: 30434

Dataset sizes:
Train: (300569, 121)
Validation: (64153, 121)
Test: (64226, 121)

Unique stays:
Train: 21303
Validation: 4565
Test: 4566


In [28]:
train_stays = set(groups.iloc[train_idx])
val_stays = set(groups_temp.iloc[val_idx])
test_stays = set(groups_temp.iloc[test_idx])

print("Train intersect Validation:", len(train_stays & val_stays))
print("Train intersect Test:", len(train_stays & test_stays))
print("Validation intersect Test:", len(val_stays & test_stays))

Train intersect Validation: 0
Train intersect Test: 0
Validation intersect Test: 0


## 18. Save final train/validation/test datasets

In [29]:
train_dataset = X_train.copy()
train_dataset[target_column] = y_train.values

validation_dataset = X_val.copy()
validation_dataset[target_column] = y_val.values

test_dataset = X_test.copy()
test_dataset[target_column] = y_test.values

train_path = "../data/processed/sepsis_progression_train.csv"
val_path = "../data/processed/sepsis_progression_validation.csv"
test_path = "../data/processed/sepsis_progression_test.csv"

train_dataset.to_csv(train_path, index=False)
validation_dataset.to_csv(val_path, index=False)
test_dataset.to_csv(test_path, index=False)

print("Datasets saved successfully!")
print("Train:", train_path, train_dataset.shape)
print("Validation:", val_path, validation_dataset.shape)
print("Test:", test_path, test_dataset.shape)

Datasets saved successfully!
Train: ../data/processed/sepsis_progression_train.csv (300569, 122)
Validation: ../data/processed/sepsis_progression_validation.csv (64153, 122)
Test: ../data/processed/sepsis_progression_test.csv (64226, 122)
